# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset investigates adoption predictors of indigenous and modern knowledge in rangeland management practices among pastoralist households in Northern Kenya.

### Dataset Source
The dataset's Croissant schema is available at the following URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` and other required libraries are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and fields using their `@id` as references. This step helps to understand the structure of the loaded dataset and which record sets and fields are available for processing.

Below, we print all available record sets, and then enumerate their fields, referencing all by their unique Croissant `@id`.

In [ ]:
# List all record sets in the dataset and their fields (by @id)
record_sets = dataset.record_sets

if not record_sets:
    print('No record sets are defined directly in the metadata. Attempting to infer from the data files...')
# If record set information is missing from top-level, mlcroissant may still expose them after parsing
else:
    print("Available Record Sets:")
    for rec in record_sets:
        print(f"  - @id: {rec['@id']}, name: {rec.get('name', '<No name>')}")
        print("    Fields:")
        for field in rec['fields']:
            print(f"      - @id: {field['@id']}, name: {field.get('name', '<No name>')}")

# Try listing all available record set @id's detected by the Dataset object
print("\nAuto-discovered record set @id's:")
rs_ids = dataset.record_set_ids()
for rsid in rs_ids:
    print(f"  - {rsid}")
    rs = dataset.record_set_by_id(rsid)
    # Print all field @id's for each record set
    if hasattr(rs, 'fields'):
        print("    Fields (by @id):")
        for field in rs.fields:
            print(f"      - {field['@id']}")

## 3. Data Extraction
Load data from each detected record set into separate DataFrames for exploration. We reference record sets and fields **exclusively by their `@id`** as per best Croissant practices.

In [ ]:
# Get all record set @id's
record_set_ids = dataset.record_set_ids()
if not record_set_ids:
    raise RuntimeError("No record sets were discovered. Unable to proceed with extraction.")

# Read all record sets using their @id, storing DataFrames keyed by record_set_id
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns in {record_set_id}: {df.columns.tolist()}")
    print(f"  Sample data:\n{df.head(2)}\n")

# Choose the main record set for further analysis (take the first one for this demo)
main_record_set_id = record_set_ids[0]
print(f"Columns available in main record set ({main_record_set_id}):\n",
      dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now we examine and process fields in the primary record set. We demonstrate filtering by a numeric field, normalizing its values, and grouping by a categorical (grouping) field—all using **`@id`** column names.

**Instructions:**
- Inspect the columns list above. Assign below appropriate `@id` values for one numeric field and one grouping (categorical) field as present in the DataFrame. For this demo, we will select the first numeric and one grouping column, if available.

In [ ]:
main_df = dataframes[main_record_set_id]

# Heuristic: Find the first numeric field and one categorical field by dtype
numeric_candidates = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
group_candidates = [col for col in main_df.columns if pd.api.types.is_object_dtype(main_df[col])]

if not numeric_candidates:
    raise RuntimeError("No numeric field found in main record set for EDA.")
numeric_field_id = numeric_candidates[0]
print(f"Using numeric field (by @id): {numeric_field_id}")

filtered_df = main_df[main_df[numeric_field_id] > main_df[numeric_field_id].mean()]
print(f"Filtered records (where {numeric_field_id} > mean): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize the numeric column
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nFirst few normalized {numeric_field_id} values:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Grouping by {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    print("No suitable grouping field detected.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship to a grouping variable (if available).

In [ ]:
# Distribution plot of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of numeric field (@id: {numeric_field_id})")
plt.xlabel(f"{numeric_field_id}")
plt.show()

# Relationship between group and mean(normalized_numeric_field), if grouping exists
if group_candidates:
    plt.figure(figsize=(10,6))
    sns.barplot(
        data=filtered_df,
        x=group_field_id,
        y=f"{numeric_field_id}_normalized",
        ci=None
    )
    plt.title(f"Mean normalized {numeric_field_id} by {group_field_id}")
    plt.xlabel(f"{group_field_id} (@id)")
    plt.ylabel(f"Normalized {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset via its Croissant schema using the `mlcroissant` library. We identified record sets and fields by their `@id`, loaded tabular data into DataFrames, performed basic filtering and normalization, and visualized key numeric trends.

- To further your analysis, examine additional fields and record sets by their `@id` (as shown), and adapt visualizations to your research needs.
- For production applications, always verify field semantics by referencing the Croissant schema documentation and original dataset codebook.